# README Técnico de Handoff — EMS MVP em AWS EKS

## Objetivo

Este notebook é o documento de handoff entre o time de desenvolvimento e o time de infraestrutura/operações.

Ele descreve:

- o que já está pronto na aplicação
- o que a infraestrutura precisa provisionar
- como publicar e atualizar a aplicação
- como operar múltiplos ambientes
- como organizar os artefatos do projeto
- como executar rollback e validações básicas

Este documento não substitui a documentação detalhada de arquitetura.  
Ele serve como **ponte operacional** para implantação.


# 1. Quando usar cada artefato

## Terraform

Use o Terraform quando o objetivo for **provisionar ou alterar infraestrutura AWS**, por exemplo:

- criar EKS
- criar node groups
- criar ECR
- criar IAM policy
- criar IRSA
- criar recursos base por ambiente

## Helm

Use Helm quando o objetivo for **implantar ou atualizar a aplicação no Kubernetes**, por exemplo:

- mudar a imagem
- mudar recursos CPU/memória
- mudar variáveis de ambiente
- mudar réplicas
- atualizar o Service

## update_app_only.sh

Use esse script quando:

- a infraestrutura já estiver pronta
- o cluster já existir
- o ECR já existir
- a ServiceAccount/IRSA já estiverem configurados
- o objetivo for apenas subir uma nova versão da aplicação


# 2. Fluxos operacionais recomendados

## Fluxo A — criação completa do ambiente

Esse fluxo é responsabilidade típica do time de infraestrutura.

### Etapas

1. aplicar Terraform por ambiente
2. validar cluster, nodegroup, ECR e IRSA
3. entregar contexto operacional ao time de desenvolvimento

### Comandos

```bash
terraform init
terraform plan -var-file=terraform-dev.tfvars -out dev.tfplan
terraform apply dev.tfplan
```

---

## Fluxo B — atualização da aplicação com infra já existente

Esse fluxo pode ser executado por desenvolvimento, DevOps ou CI/CD.

### Etapas

1. build da imagem
2. push no ECR
3. `helm upgrade --install`

### Comando

```bash
./update_app_only.sh ems_mvp.env
```


# 3. Estratégia de ambientes

## Recomendação

Use ambientes separados para:

- `dev`
- `qa`
- `prod`

## O que deve variar por ambiente

### Terraform
- nome do cluster
- nome do nodegroup
- tamanho do nodegroup
- nome do repositório ECR
- namespace Kubernetes

### Helm
- replica count
- CPU/memória
- imagem/tag
- variáveis da aplicação
- service account

## Arquivos já preparados

### Terraform
- `terraform-dev.tfvars`
- `terraform-qa.tfvars`
- `terraform-prod.tfvars`

### Helm
- `values-dev.yaml`
- `values-qa.yaml`
- `values-prod.yaml`


# 4. Estrutura recomendada de pastas

## Estrutura 

```text
ems-mvp/
├── app/
│   └── código da aplicação
├── Dockerfile
├── requirements.txt
├── certs/
│   └── apenas artefatos locais, sem versionar CA corporativa
├── docs/
│   ├── arquitetura/
│   │   ├── guia_implantacao_eks.ipynb
│   │   └── readme_handoff_infra.ipynb
│   └── troubleshooting/
│       └── problemas_comuns.md
├── deploy/
│   ├── terraform/
│   │   ├── providers.tf
│   │   ├── variables.tf
│   │   ├── main.tf
│   │   ├── outputs.tf
│   │   ├── terraform-dev.tfvars
│   │   ├── terraform-qa.tfvars
│   │   └── terraform-prod.tfvars
│   ├── helm/
│   │   └── ems-mvp-chart/
│   ├── scripts/
│   │   ├── update_app_only.sh
│   │   ├── deploy_full.sh
│   │   └── destroy.sh
│   └── env/
│       └── ems_mvp.env.example
├── .github/
│   └── workflows/
│       └── deploy.yml
└── README.md
```



## 5. Recomendação  para o projeto

- `README.md` na raiz → visão executiva e ponte para os outros documentos
- `docs/arquitetura/guia_implantacao_eks.ipynb` → guia completo
- `docs/arquitetura/readme_handoff_infra.ipynb` → handoff para o time de infraestrutura


# 6. O que o time de infra precisa receber

## Entregáveis 

1. código da aplicação
2. Dockerfile funcional
3. chart Helm
4. Terraform base
5. arquivos tfvars por ambiente
6. script de update-only
7. pipeline CI/CD
8. documentação de handoff

## Informações documentadas

- região AWS
- account id alvo
- nomes de cluster por ambiente
- nomes de namespace
- nome da ServiceAccount
- nome do repositório ECR
- porta da aplicação
- endpoint principal
- variáveis de ambiente necessárias
- Bedrock KB ID
- dependência de certificado corporativo, se houver


# 7. Fluxo de responsabilidade entre times

## Desenvolvimento

Responsável por:

- código da aplicação
- Dockerfile
- chart Helm
- variáveis da aplicação
- testes funcionais
- atualização da imagem

## Infraestrutura / Plataforma

Responsável por:

- provisionamento do cluster
- padrões de VPC/subnets
- IAM corporativo
- backend remoto do Terraform
- guardrails e segurança
- observabilidade e padrões operacionais

## Responsabilidade compartilhada

- estratégia de rollout
- política de rollback
- naming conventions
- versionamento dos valores por ambiente


# 8. Rollback operacional

## Situação típica

Uma nova imagem foi enviada ao ECR e o deploy causou erro.

## Rollback com Helm

Ver histórico:

```bash
helm history ems-mvp -n <namespace>
```

Voltar para revisão anterior:

```bash
helm rollback ems-mvp <REVISION> -n <namespace>
```

## Rollback de imagem explícita

```bash
helm upgrade --install ems-mvp ./helm_chart   -n <namespace>   -f helm_chart/values-<env>.yaml   --set image.tag=<tag-anterior>
```

## Validações após rollback

```bash
kubectl get pods -n <namespace>
kubectl logs -n <namespace> -l app=ems-mvp --tail=100
kubectl get svc -n <namespace>
```


# 9. Checklist de validação pós-deploy

## Infra

- [ ] cluster ativo
- [ ] nodegroup ativo
- [ ] nodes `Ready`
- [ ] OIDC associado
- [ ] role IRSA criada
- [ ] service account anotada

## Aplicação

- [ ] imagem publicada no ECR
- [ ] deployment aplicado
- [ ] pod `Running`
- [ ] service criado
- [ ] endpoint do load balancer publicado
- [ ] chamada ao endpoint funcionando

## Segurança

- [ ] sem access keys no manifest
- [ ] policy IAM anexada corretamente
- [ ] namespace correto
- [ ] recursos CPU/memória definidos


# 10.  organização

### Na raiz do projeto
- `README.md` curto
- código da aplicação
- Dockerfile

### Em `deploy/`
- Terraform
- Helm
- scripts
- env examples

### Em `docs/`
- documentação de arquitetura
- handoff para infra
- troubleshooting
- runbooks

## Resultado

Isso deixa o repositório:

- compreensível para dev
- útil para infra
- fácil de versionar
- pronto para CI/CD
